In [2]:
import gymnasium as gym
env = gym.make("LunarLander-v3")
obs, _ = env.reset()
print("Obs shape:", obs.shape)
print("Action space:", env.action_space.n)
env.close()

Obs shape: (8,)
Action space: 4


In [3]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

class QNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.ReLU(),
            nn.Linear(128, 128),     nn.ReLU(),
            nn.Linear(128, act_dim)
        )

    def forward(self, x):
        return self.net(x)

class ReplayBuffer:
    def __init__(self, capacity):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s_, done):
        self.buf.append((s, a, r, s_, done))

    def sample(self, n):
        batch = random.sample(self.buf, n)
        s, a, r, s_, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)),
                torch.LongTensor(a),
                torch.FloatTensor(r),
                torch.FloatTensor(np.array(s_)),
                torch.FloatTensor(d))

    def __len__(self):
        return len(self.buf)

# quick sanity check
net = QNetwork(8, 4)
dummy = torch.FloatTensor(np.random.randn(8))
print("Q-values for dummy obs:", net(dummy).detach().numpy())
print("Buffer and network OK")

Q-values for dummy obs: [0.24886528 0.02795496 0.14134562 0.00982768]
Buffer and network OK


In [4]:
import gymnasium as gym

# Hyperparameters
EPISODES      = 600
GAMMA         = 0.99
LR            = 1e-3
BATCH_SIZE    = 64
BUFFER_SIZE   = 50_000
EPS_START     = 1.0
EPS_END       = 0.01
EPS_DECAY     = 0.995
TARGET_UPDATE = 10

env       = gym.make("LunarLander-v3")
obs_dim   = env.observation_space.shape[0]
act_dim   = env.action_space.n

online_net = QNetwork(obs_dim, act_dim)
target_net = QNetwork(obs_dim, act_dim)
target_net.load_state_dict(online_net.state_dict())
target_net.eval()

optimizer = optim.Adam(online_net.parameters(), lr=LR)
buffer    = ReplayBuffer(BUFFER_SIZE)
epsilon   = EPS_START

dqn_rewards = []

for ep in range(1, EPISODES + 1):
    obs, _ = env.reset()
    ep_reward = 0
    done = False

    while not done:
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = online_net(torch.FloatTensor(obs)).argmax().item()

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.push(obs, action, reward, next_obs, float(done))
        obs = next_obs
        ep_reward += reward

        if len(buffer) >= BATCH_SIZE:
            s, a, r, s_, d = buffer.sample(BATCH_SIZE)
            with torch.no_grad():
                target = r + GAMMA * target_net(s_).max(1)[0] * (1 - d)
            current = online_net(s).gather(1, a.unsqueeze(1)).squeeze()
            loss = nn.MSELoss()(current, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(EPS_END, epsilon * EPS_DECAY)
    dqn_rewards.append(ep_reward)

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(online_net.state_dict())

    if ep % 50 == 0:
        mean100 = np.mean(dqn_rewards[-100:])
        print(f"[DQN] Ep {ep:4d} | Mean(100): {mean100:7.1f} | Epsilon: {epsilon:.3f}")

env.close()
dqn_best = np.mean(dqn_rewards[-100:])
print(f"\nDQN done. Final Mean(100): {dqn_best:.1f}")

[DQN] Ep   50 | Mean(100):  -146.2 | Epsilon: 0.778
[DQN] Ep  100 | Mean(100):  -129.5 | Epsilon: 0.606
[DQN] Ep  150 | Mean(100):   -85.9 | Epsilon: 0.471
[DQN] Ep  200 | Mean(100):   -70.1 | Epsilon: 0.367
[DQN] Ep  250 | Mean(100):   -76.1 | Epsilon: 0.286
[DQN] Ep  300 | Mean(100):   -82.6 | Epsilon: 0.222
[DQN] Ep  350 | Mean(100):   -84.7 | Epsilon: 0.173
[DQN] Ep  400 | Mean(100):   -26.0 | Epsilon: 0.135
[DQN] Ep  450 | Mean(100):    28.2 | Epsilon: 0.105
[DQN] Ep  500 | Mean(100):    41.8 | Epsilon: 0.082
[DQN] Ep  550 | Mean(100):    54.3 | Epsilon: 0.063
[DQN] Ep  600 | Mean(100):    48.6 | Epsilon: 0.049

DQN done. Final Mean(100): 48.6


In [7]:
from torch.distributions import Categorical

# PPO Hyperparameters
PPO_EPISODES   = 600
GAE_LAMBDA     = 0.95
CLIP_EPS       = 0.2
PPO_LR         = 3e-4
PPO_EPOCHS     = 4
ROLLOUT_STEPS  = 512
MINI_BATCH     = 64
VF_COEF        = 0.5
ENT_COEF       = 0.01

class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, 128),     nn.Tanh(),
        )
        self.actor  = nn.Linear(128, act_dim)
        self.critic = nn.Linear(128, 1)

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    def get_action(self, obs):
        logits, value = self.forward(obs)
        dist   = Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), value.squeeze()

    def evaluate(self, obs, actions):
        logits, value = self.forward(obs)
        dist = Categorical(logits=logits)
        return dist.log_prob(actions), dist.entropy(), value.squeeze()

def compute_gae(rewards, values, dones, last_value):
    advantages = []
    gae = 0
    values = values + [last_value]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + GAMMA * values[t+1] * (1 - dones[t]) - values[t]
        gae   = delta + GAMMA * GAE_LAMBDA * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    returns = [adv + val for adv, val in zip(advantages, values[:-1])]
    return advantages, returns

env_ppo   = gym.make("LunarLander-v3")
model     = ActorCritic(obs_dim, act_dim)
optimizer_ppo = optim.Adam(model.parameters(), lr=PPO_LR)

ppo_rewards = []
ep_reward   = 0
ep_count    = 0
obs, _      = env_ppo.reset()

obs_buf, act_buf, logp_buf = [], [], []
rew_buf, val_buf, don_buf  = [], [], []
step = 0

while ep_count < PPO_EPISODES:
    obs_t = torch.FloatTensor(obs)
    action, logp, _, value = model.get_action(obs_t)

    next_obs, reward, terminated, truncated, _ = env_ppo.step(action.item())
    done = terminated or truncated

    obs_buf.append(obs)
    act_buf.append(action.item())
    logp_buf.append(logp.item())
    rew_buf.append(reward)
    val_buf.append(value.item())
    don_buf.append(float(done))

    ep_reward += reward
    obs = next_obs
    step += 1

    if done:
        ppo_rewards.append(ep_reward)
        ep_count += 1
        ep_reward = 0
        obs, _ = env_ppo.reset()

        if ep_count % 50 == 0:
            mean100 = np.mean(ppo_rewards[-100:])
            print(f"[PPO] Ep {ep_count:4d} | Mean(100): {mean100:7.1f}")

    if step % ROLLOUT_STEPS == 0:
        with torch.no_grad():
            _, _, _, last_val = model.get_action(torch.FloatTensor(obs))

        advantages, returns = compute_gae(rew_buf, val_buf, don_buf, last_val.item())

        adv_t  = torch.FloatTensor(advantages)
        adv_t  = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
        obs_t  = torch.FloatTensor(np.array(obs_buf))
        act_t  = torch.LongTensor(act_buf)
        logp_t = torch.FloatTensor(logp_buf)
        ret_t  = torch.FloatTensor(returns)

        indices = np.arange(len(obs_buf))
        for _ in range(PPO_EPOCHS):
            np.random.shuffle(indices)
            for start in range(0, len(obs_buf), MINI_BATCH):
                idx = indices[start:start+MINI_BATCH]
                new_logp, entropy, new_val = model.evaluate(obs_t[idx], act_t[idx])

                ratio      = (new_logp - logp_t[idx]).exp()
                surr1      = ratio * adv_t[idx]
                surr2      = torch.clamp(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * adv_t[idx]
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss= nn.MSELoss()(new_val, ret_t[idx])
                ent_loss   = -entropy.mean()

                loss = actor_loss + VF_COEF * critic_loss + ENT_COEF * ent_loss
                optimizer_ppo.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer_ppo.step()

        obs_buf, act_buf, logp_buf = [], [], []
        rew_buf, val_buf, don_buf  = [], [], []

env_ppo.close()
ppo_best = np.mean(ppo_rewards[-100:])
print(f"\nPPO done. Final Mean(100): {ppo_best:.1f}")

[PPO] Ep   50 | Mean(100):  -205.2
[PPO] Ep  100 | Mean(100):  -201.8
[PPO] Ep  150 | Mean(100):  -200.4
[PPO] Ep  200 | Mean(100):  -189.4
[PPO] Ep  250 | Mean(100):  -189.8
[PPO] Ep  300 | Mean(100):  -202.8
[PPO] Ep  350 | Mean(100):  -178.1
[PPO] Ep  400 | Mean(100):  -167.0
[PPO] Ep  450 | Mean(100):  -189.7
[PPO] Ep  500 | Mean(100):  -205.7
[PPO] Ep  550 | Mean(100):  -200.6
[PPO] Ep  600 | Mean(100):  -203.9

PPO done. Final Mean(100): -203.9
